In [ ]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Cargar el dataset
ruta_dataset = "../../Limpieza/data/df_unificado_limpio_imputado.csv"
df = pd.read_csv(ruta_dataset)

In [9]:
# 1. Primero hacer el one-hot encoding
df = pd.get_dummies(df, columns=['SEGMENTO', 'TECNOLOGÍA'])

In [10]:
# 2. Características temporales
df['Año_Trimestre'] = df['AÑO'] + (df['TRIMESTRE'] - 1) / 4

In [11]:
# 3. Características de velocidad
df['Ratio_Velocidad'] = df['VELOCIDAD BAJADA'] / df['VELOCIDAD SUBIDA']
df['Velocidad_Total'] = df['VELOCIDAD BAJADA'] + df['VELOCIDAD SUBIDA']
df['Indice_Velocidad'] = np.sqrt(df['VELOCIDAD BAJADA'] * df['VELOCIDAD SUBIDA'])

In [12]:
# 4. Características geográficas
df['Distancia_Capital'] = np.sqrt(df['Latitud']**2 + df['Longitud']**2)
df['Area_Cobertura'] = abs(df['Latitud'] * df['Longitud'])

In [13]:
# 5. Características de mercado
df['Total_Accesos_Municipio'] = df.groupby(['DEPARTAMENTO', 'MUNICIPIO', 'AÑO', 'TRIMESTRE'])['No. ACCESOS FIJOS A INTERNET'].transform('sum')
df['Market_Share'] = df['No. ACCESOS FIJOS A INTERNET'] / df['Total_Accesos_Municipio']

In [14]:
# 6. Ahora sí, características de tecnología (después del one-hot encoding)
tecnologias_fibra = ['TECNOLOGÍA_FIBER TO THE HOME (FTTH)', 'TECNOLOGÍA_FIBER TO THE PREMISES',
                     'TECNOLOGÍA_FIBER TO THE BUILDING O FIBER TO THE BASEMENT (FTTB)']
df['Usa_Fibra'] = df[tecnologias_fibra].any(axis=1).astype(int)

In [15]:
# 7. Características de segmento
df['Total_Estratos'] = df[[col for col in df.columns if 'ESTRATO' in col]].sum(axis=1)
df['Indice_Estratos'] = df[[f'SEGMENTO_RESIDENCIAL - ESTRATO {i}' for i in range(1,7)]].dot(range(1,7))

In [16]:
# 8. Densidad y concentración
df['Densidad_Accesos'] = df.groupby(['DEPARTAMENTO', 'MUNICIPIO', 'AÑO', 'TRIMESTRE'])['No. ACCESOS FIJOS A INTERNET'].transform('sum') / df['Area_Cobertura']
df['Concentracion_Mercado'] = df.groupby(['DEPARTAMENTO', 'MUNICIPIO', 'AÑO', 'TRIMESTRE'])['Market_Share'].transform(lambda x: (x**2).sum())  # HHI simplificado

In [17]:
# Limpiar infinitos y nulos
for col in df.select_dtypes(include=[np.number]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].fillna(df[col].mean())

In [19]:
# Guardar el dataset con las nuevas características
ruta_salida = "../../Limpieza/data/df_mejorado_ing_caract_2.csv"
df.to_csv(ruta_salida, index=False)

In [20]:
# Mostrar las nuevas características
nuevas_caracteristicas = ['Año_Trimestre', 'Ratio_Velocidad', 'Velocidad_Total', 'Indice_Velocidad',
                          'Distancia_Capital', 'Area_Cobertura', 'Market_Share', 'Usa_Fibra',
                          'Tasa_Crecimiento', 'Promedio_Movil', 'Variacion_Trimestral']

In [21]:
print("\nEstadísticas de las nuevas características:")
print(df[nuevas_caracteristicas].describe())


Estadísticas de las nuevas características:
       Año_Trimestre  Ratio_Velocidad  Velocidad_Total  Indice_Velocidad  \
count  917885.000000    917885.000000     9.178850e+05      9.178850e+05   
mean     2022.887474         5.786512     3.462142e+02      1.570842e+02   
std         0.731218        10.669926     1.245437e+04      6.179657e+03   
min      2021.500000         0.000000     0.000000e+00      0.000000e+00   
25%      2022.250000         1.000000     1.300000e+01      6.000000e+00   
50%      2023.000000         2.500000     6.000000e+01      2.121320e+01   
75%      2023.500000         8.333333     2.250000e+02      7.071068e+01   
max      2025.364396      2344.000000     5.120000e+06      2.560000e+06   

       Distancia_Capital  Area_Cobertura   Market_Share      Usa_Fibra  \
count      917885.000000   917885.000000  917885.000000  917885.000000   
mean           75.092234      462.287727       0.002549       0.249379   
std             1.221748      194.570951       0